<h3> Parsing </h3>

In [ ]:
from unstructured.partition.text import partition_text #https://docs.unstructured.io/open-source/core-functionality/partitioning
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx

from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from unstructured.chunking.title import chunk_by_title #https://docs.unstructured.io/open-source/core-functionality/chunking
from unstructured.chunking.basic import chunk_elements


In [ ]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 1000)


    elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 1000)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 1000)


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name
                }
            )
        )
    


In [ ]:
documentos

<h3> Embeddings </h3>

In [ ]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

In [ ]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

<h3> Retrieval</h3>

<h5> Sparse Retrieval </h5>

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 10)

<h5> Dense Retrieval </h5>

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

<h5> Reciprocal Rank Fusion </h5>

In [ ]:
def Reciprocal_Rank_Fusion (rankings, k = 50): #60 é uma variável usada para reduzir o impacto de diferenças pequenas entre ranks
    
    points = {}
    title_doc = {}

    for ranking in rankings:
        #print (ranking)
        for rank, doc in enumerate (ranking):
            
            doc_content = doc.page_content
            id_doc = doc.metadata["title"]
            

            points[doc_content] = points.get(doc_content, 0) + 1 / (k + rank + 1)

            title_doc[doc_content] = id_doc
    
    ranking_final = sorted(points.items(), key = lambda x : x[1], reverse = True)


            #points[doc_content] = points.get(doc_content, 0) + 1 / (k + rank + 1)
    #Mudanças para poder englobar o nome dos docs. Importante para calcular Recall@K
    return [
        (content, score, title_doc[content])
        for content, score in ranking_final
    ]
    # return sorted (points.items(), key = lambda x: x[1], reverse = False) #Dá return dos docs ordenados e respetivas pontuações sorted

In [52]:
### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query)
d_retrieval = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])



In [ ]:
teste

<h3> Reranking </h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder = AutoModelForSequenceClassification.from_pretrained (path)
ce_tokenization = AutoTokenizer.from_pretrained (path)

In [53]:
#### teste

#features = [query] * len (teste), [t[0] for t in teste]

QUERY = [query] * len (teste)
DOCSS = [t[0] for t in teste]
DOC_NAMES = [t[2] for t in teste]
#print (QUERY)
#print (DOCSS)
#print (DOC_NAMES)


inputs = ce_tokenization (QUERY, DOCSS, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder.eval()
with torch.no_grad():
    logits = cross_encoder (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes
rerank = sorted(zip(DOCSS, DOC_NAMES, logits.tolist()), key = lambda x: x[2], reverse = True)

display (rerank)

[('A Rede EU KIDS ONLINE e o estudo por detrás deste relatório\n\nA rede EU Kids Online, uma rede colaborativa e independente, reúne investigadores de 26 países. Sob uma perspetiva multidisciplinar e multimetodológica, investigam oportunidades, riscos e segurança de crianças e jovens na internet. Ver www.eukidsonline.net\n\nSurgida em 2006, a rede tem gerado evidência de qualidade e comparabilidade sobre as experiências digitais de crianças e jovens europeus, acompanhando os seus desenvolvimentos e procurando contribuir para políticas e práticas a nível nacional, europeu e internacional. Portugal integra-a desde o seu início, tendo participado em todos os estudos comparados e tido intervenção ativa na comunicação, disseminação e exploração de resultados. uma https://eukidsonline.fcsh.unl.pt/',
  'Crianças_e_jovens_(9-17)_e_Inteligência_Artificial_Generativa_em_Portugal.pdf',
  [7.536623001098633]),
 ('Crianças e jovens (9-17) e Inteligência Artificial Generativa em Portugal\n\nResultad

In [54]:
feed_llm = [chunk for chunk, scores, docs in rerank [:5]]

display (feed_llm)

['A Rede EU KIDS ONLINE e o estudo por detrás deste relatório\n\nA rede EU Kids Online, uma rede colaborativa e independente, reúne investigadores de 26 países. Sob uma perspetiva multidisciplinar e multimetodológica, investigam oportunidades, riscos e segurança de crianças e jovens na internet. Ver www.eukidsonline.net\n\nSurgida em 2006, a rede tem gerado evidência de qualidade e comparabilidade sobre as experiências digitais de crianças e jovens europeus, acompanhando os seus desenvolvimentos e procurando contribuir para políticas e práticas a nível nacional, europeu e internacional. Portugal integra-a desde o seu início, tendo participado em todos os estudos comparados e tido intervenção ativa na comunicação, disseminação e exploração de resultados. uma https://eukidsonline.fcsh.unl.pt/',
 'Crianças e jovens (9-17) e Inteligência Artificial Generativa em Portugal\n\nResultados nacionais dos estudos EU Kids Online, 2025\n\nCristina Ponte, Susana Batista, Estrella Luna\n\nResumo:\n\n

<h3> LLM </h3>

In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1578.02it/s]


'cpu'

In [56]:
prompt = "Em que consiste o relatório da rede EU Kids Online ?"


chat_template = f"""
<|im_start|>
És um assistente especializado em responder exclusivamente com base no contexto fornecido.

Regras:
1. Utiliza apenas a informação do Contexto.
2. Não inventes informação.
3. Não uses conhecimento externo.
4. Se a resposta não existir no contexto, responde exatamente:
   "Informação não encontrada na base de dados."
5. Responde de forma breve e direta.
6. Formata a resposta de forma clara.

Contexto:
{feed_llm}
<|im_end|>

<|im_start>
{prompt}
<|im_end|>

<|im_start|>
"""

model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens = 512)

input_length = model_inputs["input_ids"].shape[1]
response_tokens = generated_ids[0][input_length:]

print(tokenizer2.decode(response_tokens, skip_special_tokens = True))

O relatório da rede EU Kids Online aborda a utilização e experiências de crianças e jovens com IA generativa em Portugal. O estudo inclui dados dos 9 a 17 anos, entrevistando adolescentes e testemunhos de outras crianças e jovens. O documento também discute o contexto europeu e estratégias digitais para o desenvolvimento e promoção da inteligência artificial em Portugal.
